In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import json
from typing import Optional, Tuple
import os

# Set device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


### 1) Loading numpy arrays

In [2]:
TENSOR_DIR = 'tensors_gpt/'  # Update to your path

# Load metadata
with open('tensors_rep/metadata.json', 'r') as f:
    metadata = json.load(f)

# Load tensors
X_train = np.load(TENSOR_DIR + 'X_train.npy')
X_val = np.load(TENSOR_DIR + 'X_val.npy')
X_test = np.load(TENSOR_DIR + 'X_test.npy')
y_train = np.load(TENSOR_DIR + 'y_train.npy')
y_val = np.load(TENSOR_DIR + 'y_val.npy')
y_test = np.load(TENSOR_DIR + 'y_test.npy')
mask_train = np.load(TENSOR_DIR + 'mask_train.npy')
mask_val = np.load(TENSOR_DIR + 'mask_val.npy')
mask_test = np.load(TENSOR_DIR + 'mask_test.npy')

# Get feature lists
original_features = metadata['original_features']   # 15 features
gap_features = metadata['gap_features']             # 15 features
all_features = metadata['all_features']             # 30 features
T = metadata['T']

print(f"X_train: {X_train.shape}  (N, T, F)")
print(f"mask_train: {mask_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Train mortality: {y_train.mean()*100:.1f}%")
print(f"Features: {len(all_features)} total ({len(original_features)} original + {len(gap_features)} time gaps)")

X_train: (45756, 48, 30)  (N, T, F)
mask_train: (45756, 48)
y_train: (45756,)
Train mortality: 11.0%
Features: 30 total (15 original + 15 time gaps)


In [4]:
class ICUDataset(Dataset):
    def __init__(self, X, mask, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.mask = torch.tensor(mask.astype(bool), dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]

### Optional different loss function

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.float()

        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = probs * targets + (1 - probs) * (1 - targets)

        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

criterion = FocalLoss(alpha=0.5, gamma=2.0)

### 2) Creating dataloaders and criterion using WeightedLoss

In [ ]:
print(f"Train samples: {len(y_train)}, mortality: {y_train.mean()*100:.1f}%")

train_ds  = ICUDataset(X_train, mask_train, y_train)
val_ds    = ICUDataset(X_val,   mask_val,   y_val)
test_ds   = ICUDataset(X_test,  mask_test,  y_test)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# weighted loss
n_neg      = (y_train == 0).sum()
n_pos      = (y_train == 1).sum()
pos_weight = torch.tensor([(n_neg / n_pos)], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"pos_weight: {pos_weight.item():.2f}")

Train samples: 45756, mortality: 11.0%
pos_weight: 8.11


### 3) Importing model implementation

In [7]:
from model import CoI

model = CoI(
    input_dim   = 30,
    emb_dim     = 16,
    hidden_dim  = 128,
    num_heads   = 2,
    num_layers  = 8,
    output_dim  = 1,
    dropout     = 0.2,
    max_seq_len = 48,
).to(DEVICE)

In [8]:
N_FEATURES = X_train.shape[2]
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)
print(f"Model input features: {N_FEATURES}")

print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model input features: 30
Total parameters: 330,274


### 4) Defining function for training and testing

In [10]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_b, mask_b, y_b in loader:
        X_b, mask_b, y_b = X_b.to(device), mask_b.to(device), y_b.to(device)
        
        optimizer.zero_grad()
        out = model(X_b, mask=mask_b)
        loss = criterion(out, y_b.unsqueeze(1).float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    losses, probs, labels = [], [], []
    with torch.no_grad():
        for X_b, mask_b, y_b in loader:
            X_b, mask_b, y_b = X_b.to(device), mask_b.to(device), y_b.to(device)
            out = model(X_b, mask=mask_b)
            loss = criterion(out, y_b.unsqueeze(1).float())
            losses.append(loss.item())
            probs.append(torch.sigmoid(out).cpu())
            labels.append(y_b.cpu())
    probs = torch.cat(probs).numpy().flatten()
    labels = torch.cat(labels).numpy()
    auroc = roc_auc_score(labels, probs)
    return np.mean(losses), auroc

### 5) Training model

In [11]:
best_auroc = 0
patience_counter = 0

for epoch in range(1, 101):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_auroc = evaluate(model, val_loader, criterion, DEVICE)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auroc={val_auroc:.4f}")
    
    if val_auroc > best_auroc:
        best_auroc = val_auroc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
    
    if patience_counter >= 10:
        print(f"Early stopping at epoch {epoch}")
        break


Epoch  10 | train_loss=0.9081 | val_loss=0.8653 | val_auroc=0.8509
Epoch  20 | train_loss=0.8561 | val_loss=0.8382 | val_auroc=0.8568
Epoch  30 | train_loss=0.8240 | val_loss=0.8416 | val_auroc=0.8646
Epoch  40 | train_loss=0.7998 | val_loss=0.8203 | val_auroc=0.8660
Epoch  50 | train_loss=0.7811 | val_loss=0.8112 | val_auroc=0.8679
Epoch  60 | train_loss=0.7447 | val_loss=0.8321 | val_auroc=0.8669
Early stopping at epoch 67


### 6) Evaluating on test set

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report

def evaluate_test(model, test_loader, device, threshold=0.5):
    """
    Evaluate model on test set and return comprehensive metrics.
    
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader for test set
        device: torch device
        threshold: Classification threshold (default 0.5)
    
    Returns:
        dict: Dictionary containing all metrics
    """
    model.eval()
    all_probs = []
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for X_b, mask_b, y_b in test_loader:
            X_b, mask_b, y_b = X_b.to(device), mask_b.to(device), y_b.to(device)
            
            # Forward pass
            out = model(X_b, mask=mask_b)
            probs = torch.sigmoid(out).cpu().numpy().flatten()
            preds = (probs >= threshold).astype(int)
            
            all_probs.extend(probs)
            all_labels.extend(y_b.cpu().numpy())
            all_preds.extend(preds)
    
    # Convert to numpy arrays
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    
    # Calculate metrics
    metrics = {
        'AUROC': roc_auc_score(all_labels, all_probs),
        'F1': f1_score(all_labels, all_preds, zero_division=0),
        'Precision': precision_score(all_labels, all_preds, zero_division=0),
        'Recall': recall_score(all_labels, all_preds, zero_division=0),
        'Accuracy': accuracy_score(all_labels, all_preds),
        'Threshold': threshold
    }
    
    return metrics, all_probs, all_labels, all_preds


def find_optimal_threshold(model, val_loader, device):
    """
    Find optimal threshold using validation set (maximizes F1).
    
    Returns:
        float: Optimal threshold value
    """
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_b, mask_b, y_b in val_loader:
            X_b, mask_b, y_b = X_b.to(device), mask_b.to(device), y_b.to(device)
            out = model(X_b, mask=mask_b)
            probs = torch.sigmoid(out).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(y_b.cpu().numpy())
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Try different thresholds
    best_f1 = 0
    best_threshold = 0.5
    
    for threshold in np.arange(0.1, 0.9, 0.05):
        preds = (all_probs >= threshold).astype(int)
        f1 = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    print(f"Optimal threshold: {best_threshold:.3f} (F1: {best_f1:.4f})")
    return best_threshold


def print_detailed_metrics(metrics, all_labels, all_preds, all_probs):
    """Print detailed metrics including confusion matrix and classification report."""
    print("\n" + "="*60)
    print("TEST SET RESULTS")
    print("="*60)
    
    print(f"\n📊 Core Metrics:")
    print(f"   AUROC:      {metrics['AUROC']:.4f}")
    print(f"   Accuracy:   {metrics['Accuracy']:.4f}")
    print(f"   F1 Score:   {metrics['F1']:.4f}")
    print(f"   Precision:  {metrics['Precision']:.4f}")
    print(f"   Recall:     {metrics['Recall']:.4f}")
    print(f"   Threshold:  {metrics['Threshold']:.3f}")
    
    print(f"\n📈 Confusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"              Predicted")
    print(f"              Neg   Pos")
    print(f"   Actual Neg  {cm[0,0]:5d}  {cm[0,1]:5d}")
    print(f"          Pos  {cm[1,0]:5d}  {cm[1,1]:5d}")
    
    # Calculate additional metrics
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    print(f"\n📉 Additional Metrics:")
    print(f"   Specificity: {specificity:.4f}")
    print(f"   NPV:         {npv:.4f}")
    
    print(f"\n📋 Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Survived', 'Mortality'], zero_division=0))
    
    # Class distribution in test set
    n_pos = (all_labels == 1).sum()
    n_neg = (all_labels == 0).sum()
    print(f"\n📊 Test Set Distribution:")
    print(f"   Mortality:  {n_pos} ({n_pos/len(all_labels)*100:.1f}%)")
    print(f"   Survived:   {n_neg} ({n_neg/len(all_labels)*100:.1f}%)")


# ========== USAGE ==========

# After training, evaluate on test set:

# Option 1: Use default threshold 0.5
metrics, probs, labels, preds = evaluate_test(model, test_loader, DEVICE, threshold=0.5)
print_detailed_metrics(metrics, labels, preds, probs)

# Option 2: Find optimal threshold on validation set first
optimal_threshold = find_optimal_threshold(model, val_loader, DEVICE)
metrics, probs, labels, preds = evaluate_test(model, test_loader, DEVICE, threshold=optimal_threshold)
print_detailed_metrics(metrics, labels, preds, probs)

# Option 3: Get predictions for individual analysis
def get_predictions(model, loader, device):
    """Return all predictions, probabilities, and labels for further analysis."""
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_b, mask_b, y_b in loader:
            X_b, mask_b, y_b = X_b.to(device), mask_b.to(device), y_b.to(device)
            out = model(X_b, mask=mask_b)
            probs = torch.sigmoid(out).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(y_b.cpu().numpy())
    
    return np.array(all_probs), np.array(all_labels)

# Get predictions for further analysis
test_probs, test_labels = get_predictions(model, test_loader, DEVICE)
print(f"\n✓ Got {len(test_probs)} test predictions")


TEST SET RESULTS

📊 Core Metrics:
   AUROC:      0.8692
   Accuracy:   0.7056
   F1 Score:   0.3890
   Precision:  0.2518
   Recall:     0.8541
   Threshold:  0.350

📈 Confusion Matrix:
              Predicted
              Neg   Pos
   Actual Neg   5999   2730
          Pos    157    919

📉 Additional Metrics:
   Specificity: 0.6872
   NPV:         0.9745

📋 Classification Report:
              precision    recall  f1-score   support

    Survived       0.97      0.69      0.81      8729
   Mortality       0.25      0.85      0.39      1076

    accuracy                           0.71      9805
   macro avg       0.61      0.77      0.60      9805
weighted avg       0.90      0.71      0.76      9805


📊 Test Set Distribution:
   Mortality:  1076 (11.0%)
   Survived:   8729 (89.0%)
Optimal threshold: 0.750 (F1: 0.5391)

TEST SET RESULTS

📊 Core Metrics:
   AUROC:      0.8692
   Accuracy:   0.8948
   F1 Score:   0.5281
   Precision:  0.5203
   Recall:     0.5362
   Threshold:  0.750

